### Config

In [1]:
import os
from google.cloud import aiplatform
from dotenv import load_dotenv
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from google.cloud import storage
import vertexai
import numpy as np

load_dotenv() 

PROJECT_ID = os.environ["PROJECT_ID"]
LOCATION = os.environ["LOCATION"]

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"./service_account.json"

aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
)

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION
)

2026-03-04 10:20:27.751351: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Model

In [2]:
def build_model():
  model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=[10]),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
  ])

  return model

In [3]:
tf_model = build_model()

/home/ridwanfatur/learning/venv_3_11_13/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1772594459.771655    5376 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2248 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


In [5]:
tf_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,929 (19.25 KB)

 Trainable params: 4,929 (19.25 KB)

 Non-trainable params: 0 (0.00 B)

In [28]:
# example_input = np.random.randn(1, 10)
example_input = np.array([
    [1.0, 2.0, 3.0, 4.0, 1.0, 2.0, 3.0, 4.0, 9.0, 9.0],
    [1.0, 2.0, 3.0, 4.0, 1.0, 2.0, 3.0, 4.0, 9.0, 11.0],
])

In [29]:
tf_model(example_input)

<tf.Tensor: shape=(2, 1), dtype=float32, numpy=
array([[0.05298281],
       [0.10310739]], dtype=float32)>

### Export to Bucket

In [9]:
# Create Bucket
client = storage.Client()

buckets = client.list_buckets()

for bucket in buckets:
    print(bucket.name)

cvinsight-ai-bucket-project


In [10]:
bucket_name = "my-vertex-bucket-123456"

bucket = client.bucket(bucket_name)
bucket.location = "us-central1"

bucket = client.create_bucket(bucket)

/tmp/ipykernel_5376/3706126528.py:4: DeprecationWarning: Assignment to 'Bucket.location' is deprecated, as it is only valid before the bucket is created. Instead, pass the location to `Bucket.create`.
  bucket.location = "us-central1"


In [12]:
gcs_path = "gs://my-vertex-bucket-123456/tensorflow"
tf_model.export(gcs_path)

INFO:tensorflow:Assets written to: gs://my-vertex-bucket-123456/tensorflow/assets


INFO:tensorflow:Assets written to: gs://my-vertex-bucket-123456/tensorflow/assets


Saved artifact at 'gs://my-vertex-bucket-123456/tensorflow'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 10), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139617886700176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139617886703440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139617886702096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139617886700752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139617886704016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139617886700944: TensorSpec(shape=(), dtype=tf.resource, name=None)


### For Checking

In [33]:
models = aiplatform.Model.list()

for model in models:
    print("Name:", model.display_name)
    print("Resource:", model.resource_name)
    print("----")

Name: random-forest-classifier-model
Resource: projects/344969539300/locations/us-central1/models/8820070072274911232
----


In [34]:
endpoints = aiplatform.Endpoint.list()

for ep in endpoints:
    print("Name:", ep.display_name)
    print("Resource:", ep.resource_name)
    print("----")

Name: random-forest-classifier-endpoint
Resource: projects/344969539300/locations/us-central1/endpoints/5382998373114576896
----


### Upload Model and Create Endpoint

In [16]:
model = aiplatform.Model.upload(
    display_name="tensorflow-model",
    artifact_uri="gs://my-vertex-bucket-123456/tensorflow",
    serving_container_image_uri=
    "us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-13:latest"
)

Creating Model
Create Model backing LRO: projects/344969539300/locations/us-central1/models/4775415394431139840/operations/8031631870502371328
Model created. Resource name: projects/344969539300/locations/us-central1/models/4775415394431139840@1
To use this Model in another session:
model = aiplatform.Model('projects/344969539300/locations/us-central1/models/4775415394431139840@1')


In [17]:
endpoint = aiplatform.Endpoint.create(
    display_name="tensorflow-endpoint"
)

print("Endpoint created:")
print(endpoint.resource_name)

Creating Endpoint
Create Endpoint backing LRO: projects/344969539300/locations/us-central1/endpoints/1008666128798449664/operations/7726231520771309568
Endpoint created. Resource name: projects/344969539300/locations/us-central1/endpoints/1008666128798449664
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/344969539300/locations/us-central1/endpoints/1008666128798449664')
Endpoint created:
projects/344969539300/locations/us-central1/endpoints/1008666128798449664


### Deploy Endpoint

In [20]:
model = aiplatform.Model(
    "projects/344969539300/locations/us-central1/models/4775415394431139840"
)

In [22]:
model.deploy(
    endpoint=endpoint,
    deployed_model_display_name="tensorflow-model-deployed",
    machine_type="e2-standard-2",
    min_replica_count=1,
    max_replica_count=1,
)

Deploying model to Endpoint : projects/344969539300/locations/us-central1/endpoints/1008666128798449664
Deploy Endpoint model backing LRO: projects/344969539300/locations/us-central1/endpoints/1008666128798449664/operations/3587423463217823744
Endpoint model deployed. Resource name: projects/344969539300/locations/us-central1/endpoints/1008666128798449664


resource name: projects/344969539300/locations/us-central1/endpoints/1008666128798449664

### Inference

In [23]:
endpoint = aiplatform.Endpoint(
    "projects/344969539300/locations/us-central1/endpoints/1008666128798449664"
)

In [24]:
instances = [
    [1.0, 2.0, 3.0, 4.0, 1.0, 2.0, 3.0, 4.0, 9.0, 9.0],
    [1.0, 2.0, 3.0, 4.0, 1.0, 2.0, 3.0, 4.0, 9.0, 11.0],
]

response = endpoint.predict(instances=instances)

In [25]:
response

Prediction(predictions=[[0.0529830456], [0.103107452]], deployed_model_id='3171502257657085952', metadata=None, model_version_id='1', model_resource_name='projects/344969539300/locations/us-central1/models/4775415394431139840', explanations=None)

### Clean

In [30]:
# Clean
endpoint = aiplatform.Endpoint(
    "projects/344969539300/locations/us-central1/endpoints/1008666128798449664"
)

endpoint.delete(force=True)

model = aiplatform.Model(
    "projects/344969539300/locations/us-central1/models/4775415394431139840"
)

model.delete()

Undeploying Endpoint model: projects/344969539300/locations/us-central1/endpoints/1008666128798449664
Undeploy Endpoint model backing LRO: projects/344969539300/locations/us-central1/endpoints/1008666128798449664/operations/4051434962325340160
Endpoint model undeployed. Resource name: projects/344969539300/locations/us-central1/endpoints/1008666128798449664
Deleting Endpoint : projects/344969539300/locations/us-central1/endpoints/1008666128798449664
Endpoint deleted. . Resource name: projects/344969539300/locations/us-central1/endpoints/1008666128798449664
Deleting Endpoint resource: projects/344969539300/locations/us-central1/endpoints/1008666128798449664
Delete Endpoint backing LRO: projects/344969539300/locations/us-central1/operations/4779892202052517888
Endpoint resource projects/344969539300/locations/us-central1/endpoints/1008666128798449664 deleted.
Deleting Model : projects/344969539300/locations/us-central1/models/4775415394431139840
Model deleted. . Resource name: projects/3